In [1]:
import tensorflow as tf
import keras
import os

2026-02-10 14:05:01.512751: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-10 14:05:01.518932: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770732301.526494 2763480 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770732301.528819 2763480 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-10 14:05:01.536467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
gpus = tf.config.list_physical_devices(device_type='GPU')
print(gpus)
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    tf.config.set_visible_devices(gpus[0], 'GPU')
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
  except RuntimeError as e:
    print(e)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
2 Physical GPUs, 1 Logical GPUs


I0000 00:00:1770732324.294550 2763480 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1608 MB memory:  -> device: 0, name: NVIDIA RTX A4000, pci bus id: 0000:01:00.0, compute capability: 8.6


In [3]:
(training_images, training_labels) , \
(validation_images, validation_labels) = \
tf.keras.datasets.cifar10.load_data()

In [4]:
def preprocess_image_input(input_images):
  return input_images.astype('float32')

train_X = preprocess_image_input(training_images)
valid_X = preprocess_image_input(validation_images)

In [5]:
@keras.saving.register_keras_serializable()
class ResizeLayer(tf.keras.layers.Layer):
  def __init__(self, size, method='nearest', **kwargs):
    super(ResizeLayer, self).__init__(**kwargs)
    self.size = size
    self.method = method

  def call(self, inputs):
    return tf.image.resize(inputs, size=self.size, method=self.method)

'''
Feature Extraction is performed by EfficientNetB0 pretrained on imagenet weights. 
Input size is 224 x 224.
'''
def feature_extractor(inputs):
  feature_extractor = tf.keras.applications.EfficientNetB0(input_shape=(224, 224, 3),
                                               include_top=False,
                                               weights='imagenet')(inputs)
  return feature_extractor


'''
Defines final dense layers and subsequent softmax layer for classification.
'''
def classifier(inputs):
  x = tf.keras.layers.GlobalAveragePooling2D()(inputs)
  x = tf.keras.layers.Dropout(0.2)(x)
  x = tf.keras.layers.Flatten()(x)
  #x = tf.keras.layers.Dense(512, activation="relu")(x)
  #x = tf.keras.layers.Dropout(0.5)(x)
  x = tf.keras.layers.Dense(128, activation="relu")(x)
  x = tf.keras.layers.Dropout(0.5)(x)
  x = tf.keras.layers.Dense(10, activation="softmax", name="classification")(x)
  return x

'''
Since input image size is (32 x 32), first upsample the image by factor of (7x7) to transform it to (224 x 224)
Connect the feature extraction and "classifier" layers to build the model.
'''
def final_model(inputs):
  resize = ResizeLayer(size=(inputs.shape[1] * 7, inputs.shape[2] * 7))(inputs)
  resnet_feature_extractor = feature_extractor(resize)
  classification_output = classifier(resnet_feature_extractor)
  return classification_output

'''
Define the model and compile it. 
Use Stochastic Gradient Descent as the optimizer.
Use Sparse Categorical CrossEntropy as the loss function.
'''
def define_compile_model():
  inputs = tf.keras.layers.Input(shape=(32,32,3))
  classification_output = final_model(inputs) 
  model = tf.keras.Model(inputs=inputs, outputs = classification_output)
 
  model.compile(optimizer='SGD', 
                loss='sparse_categorical_crossentropy',
                metrics = ['accuracy'])
  
  return model

model = define_compile_model()

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resize_layer (ResizeLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification (Dense)          │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,829 (16.08 MB)

 Trainable params: 4,172,806 (15.92 MB)

 Non-trainable params: 42,023 (164.16 KB)

In [ ]:
# Traint the model (only for Cifar10 dataset)
EPOCHS = 3

history = model.fit(train_X, 
                    training_labels, 
                    epochs=EPOCHS, 
                    validation_data = (valid_X, validation_labels), 
                    batch_size=64)

Epoch 1/15


I0000 00:00:1770732359.531436 2763889 service.cc:148] XLA service 0x7efac8010b00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1770732359.531590 2763889 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4000, Compute Capability 8.6
2026-02-10 14:05:59.776502: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1770732360.413880 2763889 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-02-10 14:06:01.627881: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_16844', 96 bytes spill stores, 96 bytes spill loads

2026-02-10 14:06:01.641592: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_16844', 24 bytes spil

UnknownError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main

  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code

  File "/home/g.perlo/jacki_venv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/g.perlo/jacki_venv/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/g.perlo/jacki_venv/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/g.perlo/jacki_venv/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.10/asyncio/base_events.py", line 603, in run_forever

  File "/usr/lib/python3.10/asyncio/base_events.py", line 1909, in _run_once

  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run

  File "/home/g.perlo/jacki_venv/lib/python3.10/site-packages/ipykernel/k

In [ ]:
MODEL_PATH = "../models_data/efficientnetB0/"
os.makedirs(MODEL_PATH, exist_ok=True)
model.save(MODEL_PATH+'efficientnetB0_fp32_cifar10.keras')

In [ ]:
model.evaluate(valid_X, validation_labels)
model.evaluate(valid_X[:2000], validation_labels[:2000])

In [ ]:
def representative_dataset():
  for images in tf.data.Dataset.from_tensor_slices(train_X).batch(1).take(1000):
    yield [images]

loaded_model = keras.saving.load_model("../models_data/efficientnetB0/efficientnetB0_fp32_cifar10.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(loaded_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] 
converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_quant_model = converter.convert()
with open('../models_data/efficientnetB0/efficientnetB0_uint8_cifar10.tflite', 'wb') as f:
  f.write(tflite_quant_model)

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_quant_model = converter.convert()
with open('../models_data/efficientnetB0/efficientnetB0_int8_cifar10.tflite', 'wb') as f:
  f.write(tflite_quant_model)